In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
import os

import gfdl_utils.core as gu
import CM4Xutils
import cftime
import numpy as np
import pandas as pd
import xarray as xr
import xgcm
import xhistogram
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 11})

In [3]:
include_piControl=False

### Load postprocessed model datasets

In [4]:
from common import *
grids = load_datasets()

No ideal age companion store for CM4Xp25 (missing ['CM4Xp25_historical_ideal_age_z.zarr', 'CM4Xp25_ssp585_ideal_age_z.zarr']); `agessc` will be absent from the CM4Xp25 dataset.
Inferring Z grid coordinate: depth `z_`


/work5/hfd/projects/CM4XTransientTracers/analysis/common.py:75: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  xr.open_zarr(f"../data/interim/{model}_historical_transient_tracers_surface.zarr"),


/work5/hfd/projects/CM4XTransientTracers/analysis/common.py:64: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  return xr.concat([xr.open_zarr(p) for p in paths], dim="year")


/work5/hfd/projects/CM4XTransientTracers/analysis/common.py:64: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  return xr.concat([xr.open_zarr(p) for p in paths], dim="year")


Inferring Z grid coordinate: depth `z_`


In [5]:
dlat = 1.
lat_bins = np.arange(-90., 90+dlat, dlat)
lat_centers = np.arange(-90.+dlat/2, 90+dlat/2, dlat)

for sim in ["CM4Xp25_forced", "CM4Xp125_forced"]:
    ds = grids[sim]._ds.sel(year=slice(1991,1997))
    ds_reduced = xr.Dataset(coords=ds.coords)
    for tr in ["cfc11", "cfc12", "sf6"]:
        ds_reduced[rf"{tr}_vertical_int"] = (ds[tr]*ds.thkcello*m2_per_km2).mean("year").sum("z_l")
        
        volcello = ds.thkcello.fillna(0.) * ds.areacello.fillna(0.)
        cfc11_content_zonally_integrated = xhistogram.xarray.histogram(
            ds.geolat.rename("lat"),
            bins=[lat_bins],
            dim=("xh", "yh", "year"),
            weights=ds[tr].fillna(0.)*volcello,
            bin_dim_suffix="",
            block_size=None,
            keep_coords=True
        )
        volume_content_zonally_integrated = xhistogram.xarray.histogram(
            ds.geolat.rename("lat"),
            bins=[lat_bins],
            dim=("xh", "yh", "year"),
            weights=volcello,
            bin_dim_suffix="",
            block_size=None,
            keep_coords=True,
        )
        ds_reduced[rf"{tr}_zonal_int"] = cfc11_content_zonally_integrated/volume_content_zonally_integrated
    
    ds_reduced.to_netcdf(f"../data/processed/tracer_integrals_{sim}.nc", mode="w")

/work/hfd/.conda/envs/CM4XTransientTracers/lib/python3.11/site-packages/dask/_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


/work/hfd/.conda/envs/CM4XTransientTracers/lib/python3.11/site-packages/dask/_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
